# Générateurs

## Transformateurs d'itérables

Les générateurs sont très utilisés pour transformer et utiliser des itérables sans jamais les stocker entièrement.

Écrivez les transformateurs suivants, qui traiteront des itérables d'entiers :

- `double` qui double les valeurs qu'on lui passe
- `filter_even` qui retire les nombres impairs de la séquence qu'on lui fournit

Testez vos transformateurs sur un grand itérable. Observez l'utilisation mémoire de l'instance.

In [ ]:
# Votre code ici

### Solution

In [ ]:
from collections.abc import Iterable, Iterator


def double(iterable: Iterable[int]) -> Iterator[int]:
  for item in iterable:
    yield item * 2


def filter_even(iterable: Iterable[int]) -> Iterator[int]:
  for item in iterable:
    if item % 2:
      yield item

for i in filter_even(double(range(100_000_000))):
    pass

## Générateur de dates infini

Écrivez la fonction `gen_dates` qui, à partir d'une date donnée, génère toutes les dates successives séparées d'une semaine.

In [ ]:
import datetime
from collections.abc import Iterator


def gen_dates(date: datetime.date) -> Iterator[datetime.date]:
  pass  # Votre code ici

Proposez une méthode directe pour utiliser ce générateur infini.

Proposez aussi une méthode `take[T](iterable: Iterable[T], n: int) -> Iterator[T]` (qui prend en entrée un itérable et renvoie ses `n` premiers éléments) pour utiliser ce générateur infini.

In [ ]:
# Votre code ici

### Solution

In [ ]:
import datetime
from collections.abc import Iterable, Iterator


def gen_dates(date: datetime.date) -> Iterator[datetime.date]:
  week = datetime.timedelta(days=7)
  while True:
    date += week
    yield date


# Avec l'opérateur walrus
def gen_dates(date: datetime.date) -> Iterator[datetime.date]:
  week = datetime.timedelta(days=7)
  while True:
    yield (date := date + week)


for i, d in enumerate(gen_dates(datetime.date(1900, 1, 1))):
  if i == 10:
    break
  print(d)


def take[T](iterable: Iterable[T], n: int) -> Iterator[T]:
  for i, item in enumerate(iterable):
    if i == n:
      break
    yield item


def take[T](iterable: Iterable[T], n: int) -> Iterator[T]:
  for item, _ in zip(iterable, range(n)):
    yield item


for date in take(gen_dates(datetime.date(1900, 1, 1)), 10):
  print(date)

## Utilisation de la méthode `send` d'un générateur (très difficile)

Concevez un générateur qui, étant donné une somme de départ d'argent, va recevoir de votre part un taux d'intérêt et vous rendre la nouvelle valeur de la somme après application du taux.

In [ ]:
# Votre code ici

### Solution

In [ ]:
from collections.abc import Generator


def interests(amount: float) -> Generator[float, float, None]:
  while True:
    rate = yield amount
    amount *= rate
    # Ou directement
    # amount *= yield amount


amounts = interests(100)
print(next(amounts))  # Initialisation du générateur
for rate in [1.05, 1.08, 0.98]:
  print(amounts.send(rate))

## Utilisation de la méthode `throw` d'un générateur (très difficile)

Repartez du code défini à l'exercice précédent et ajoutez un mécanisme pour rajouter une somme d'argent (ou en retirer avec un somme négative). Pour cela, vous pouvez utiliser `throw`, la nature de l'exception représentant un message pour le générateur.

In [ ]:
# Votre code ici

### Solution

In [ ]:
from collections.abc import Generator


class AddAmountException(Exception):
  def __init__(self, value):
    super().__init__(value)
    self.value = value


def interests(amount: float) -> Generator[float, float, None]:
  while True:
    try:
      rate = (yield amount)
      amount *= rate
    except AddAmountException as e:
      amount += e.value


amounts = interests(100)
amounts.send(None)  # Initialisation du générateur
print(amounts.send(1.05))
print(amounts.send(1.08))
amounts.throw(AddAmountException(100))
print(amounts.send(0.98))